## Prétraitement des données

Traitement des données pour visualisation

1) supprimer les tirets et la répatition de l'intitulé des variables

In [4]:
#1) supprimer les tirets et la répétition de l'intitulé des variables

input_file = "LECLERC.csv"
output_file = "LECLERC_cleaned.csv"

with open(input_file, "r", encoding="utf-8") as f_in:
    lines = f_in.readlines()

cleaned_lines = []
header_encountered = False

for line in lines:
    # Suppression des lignes composées uniquement de tirets
    if line.strip().startswith('-'):
        continue

    # Suppression des répétitions d'en-tête (c-à-d. "Platform,Product Name,...") 
    # une fois qu'on l'a déjà rencontrée
    if line.startswith("Platform") and header_encountered:
        continue
    elif line.startswith("Platform"):
        header_encountered = True

    cleaned_lines.append(line)

with open(output_file, "w", encoding="utf-8") as f_out:
    f_out.writelines(cleaned_lines)





In [5]:
import pandas as pd

df = pd.read_csv('LECLERC_cleaned.csv')
df['Price'] = (df['Price']
               .str.replace(' €', '', regex=False)  # Supprime ' €'
               .str.replace(',', '.', regex=False)  # Remplace ',' par '.'
               .astype(float))                      # Convertit en float



Ajout d'un ID pour chaque paire product-seller

In [6]:
from uuid import uuid4

#je rajoute un id pour chaque groupe product name-seller différent
def generate_uuid():
    return str(uuid4())

# Créer un dictionnaire de correspondance pour chaque combinaison unique
# On utilise groupby pour obtenir les groupes distincts et on attribue à chacun un UUID
group_mapping = {
    (row['Product Name'], row['Seller']): generate_uuid()
    for _, row in df[['Product Name', 'Seller']].drop_duplicates().iterrows()
}

# Ajouter une nouvelle colonne en mappant chaque ligne avec le UUID correspondant
df['ID'] = df.apply(lambda row: group_mapping[(row['Product Name'], row['Seller'])], axis=1)


Export des données sous format csv

In [7]:
# Écriture du DataFrame nettoyé dans un nouveau fichier CSV
df.to_csv(output_file, index=False)

## Extraction des produits de la Buy Box

On recupère les premiers produits pour chaque combinaison **TimeStamp-Product Name** qui sont les produits de la **Buy Box**.

In [8]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M:%S')
df_grouped = df.groupby(['Timestamp', 'Product Name'], as_index=False).first()

bbox = df_grouped[['Timestamp', 'Product Name', 'Seller', 'Price']]

bbox.head()

,Timestamp,Product Name,Seller,Price
0,2024-12-24 13:30:11,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",E.Leclerc,1349.0
1,2024-12-24 13:30:17,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",E.Leclerc,1098.0
2,2024-12-24 13:30:24,Non trouvé,E.Leclerc,969.0
3,2024-12-24 13:30:31,"Apple iPhone 16 Plus 17 cm (6.7"") Double SIM i...",Stock e-commerce,1580.8
4,2024-12-24 13:30:38,"Apple iPhone 16 Plus 17 cm (6.7"") Double SIM i...",E.Leclerc,1240.0


In [9]:
bbox.to_csv('BBox.csv')